# Conveyor perception

An end-to-end industrial CV pipeline on a free T4: real recycling data → trained model → live detection → triage → Coach → publish → feedback loop. Runs in ~5 min.

**Layout** — 5 sections (§1 setup, §2 walkthrough, §3 compare, §4 coach, §5 feedback loop). Each cell has a one-line name and a tight status block. Re-run any cell to retry it.

**Live state** — every `state.log` / `state.error` / `state.metric` call appends to a session log. The last 25 events render at the bottom of cell 1 — re-run that cell to refresh the view.

**Errors** — caught per-cell, never crash the notebook. The Coach in §3 diagnoses them automatically. Set `GEMINI_API_KEY` in the secrets panel for AI diagnosis (without it, you get static hints — still useful).


In [ ]:
# --- Cell 1: Runtime + env check ---
REPO = '/content/conveyor-perception'
REPO_URL = 'https://github.com/roniejosephv-star/conveyor-perception.git'
import os, sys, subprocess, shutil
from pathlib import Path

# --- Step 0: ensure CWD is a real directory (BEFORE any subprocess call) ---
# If the previous session left us in a CWD that was deleted (Colab's
# /content can be wiped on runtime restart), every subprocess.run will
# fail with `fatal: Unable to read current working directory`.
# Fix: chdir to a known-good path (/content always exists in Colab).
try:
    os.getcwd()  # raises FileNotFoundError if CWD doesn't exist
except (FileNotFoundError, OSError) as _e:
    print(f'⚠ CWD was invalid ({_e}); falling back to /content')
    os.chdir('/content')
# If /content itself is missing (rare), try the home dir as a last resort
if not os.path.isdir(os.getcwd()):
    os.chdir('/')

# --- Step 1: chdir into the repo (idempotent + safe) ---
# Tolerate any state: not-yet-cloned, file-instead-of-dir, broken repo.
try:
    if os.path.isdir(REPO):
        os.chdir(REPO)
    elif os.path.exists(REPO):
        # A file at this path (rare, but possible from a botched prior session).
        # Don't crash — the self-heal below will nuke and re-clone.
        print(f'⚠ {REPO} exists but is not a directory; will be replaced.')
except OSError as _e:
    print(f'⚠ os.chdir({REPO}) failed: {_e}; continuing without it')

# Always add the path so the import below has a chance to find colab_session
# even on a fresh session before the clone finishes.
for _p in (REPO, os.path.join(REPO, 'notebooks'), os.path.join(REPO, 'src')):
    if _p not in sys.path:
        sys.path.insert(0, _p)

# --- Step 2: helpers (locally scoped, no module-level state) ---
def _do_clone() -> bool:
    '''Nuke + re-clone. Returns True on success.'''
    try:
        if os.path.exists(REPO):
            shutil.rmtree(REPO, ignore_errors=True)
        result = subprocess.run(
            ['git', 'clone', REPO_URL, REPO],
            capture_output=True, text=True, timeout=60,
            cwd='/content',  # always clone from a known-good CWD
        )
        if result.returncode != 0:
            print(f'  ✗ git clone failed (rc={result.returncode}): {result.stderr.strip()[:200]}')
            return False
        return True
    except Exception as _e:
        print(f'  ✗ git clone raised: {type(_e).__name__}: {_e}')
        return False

def _do_pull() -> bool:
    '''Best-effort pull. Always returns True (we tolerate pull conflicts).'''
    try:
        subprocess.run(
            ['git', '-C', REPO, 'pull', '--rebase'],
            capture_output=True, text=True, timeout=30,
            cwd='/content',  # always pull from a known-good CWD
        )
        return True
    except Exception as _e:
        print(f'  ⚠ git pull raised: {type(_e).__name__}: {_e}')
        return False

# --- Step 3: self-heal the import (3 tiers) ---
_colab_session_ready = False
try:
    from colab_session import env_check, get_state, cell  # noqa: F401
    _colab_session_ready = True
except ImportError as _e:
    print(f'⚠ colab_session import failed: {_e}')
    print(f'  → self-heal: checking {REPO}...')
    repo_path = Path(REPO)
    has_git = repo_path.exists() and (repo_path / '.git').exists()
    has_file = (repo_path / 'notebooks' / 'colab_session.py').exists()
    if has_file:
        # Repo + colab_session.py exist; import failed → likely stale file
        print('  → file exists but import failed; pulling latest...')
        _do_pull()
    elif has_git:
        print('  → repo exists but colab_session.py missing; pulling...')
        _do_pull()
    elif repo_path.exists():
        print('  → bad state at REPO path; re-cloning fresh...')
        _do_clone()
    else:
        print('  → cloning repo (~5s)...')
        _do_clone()
    # Re-insert paths and try again
    for _p in (REPO, os.path.join(REPO, 'notebooks')):
        if _p not in sys.path:
            sys.path.insert(0, _p)
    try:
        if os.path.isdir(REPO):
            os.chdir(REPO)
    except OSError:
        pass
    try:
        from colab_session import env_check, get_state, cell  # noqa: F401
        _colab_session_ready = True
    except ImportError as _e2:
        # Last resort: nuke + re-clone fresh
        print(f'  ⚠ import still failed ({_e2}); nuking + re-cloning fresh...')
        if _do_clone():
            for _p in (REPO, os.path.join(REPO, 'notebooks')):
                if _p not in sys.path:
                    sys.path.insert(0, _p)
            try:
                if os.path.isdir(REPO):
                    os.chdir(REPO)
            except OSError:
                pass
            # Diagnostic: surface the actual state of the cloned file so the
            # user can see EXACTLY why the import failed (file missing,
            # 0-byte, or sys.path wrong). This is for debugging — the
            # import still runs after.
            _cs_path = os.path.join(REPO, 'notebooks', 'colab_session.py')
            print(f'  📋 Diagnostic: colab_session.py exists={os.path.exists(_cs_path)}, '
                  f'size={os.path.getsize(_cs_path) if os.path.exists(_cs_path) else 0} bytes')
            print(f'  📋 Diagnostic: sys.path[0:3]={sys.path[0:3]}')
            print(f'  📋 Diagnostic: REPO contents={os.listdir(REPO)[:5] if os.path.isdir(REPO) else "missing"}')
            _nb_dir = os.path.join(REPO, 'notebooks')
            print(f'  📋 Diagnostic: notebooks/ contents={os.listdir(_nb_dir)[:5] if os.path.isdir(_nb_dir) else "missing"}')
            # Sanity check: try to compile the file directly. If it
            # fails to compile, Python hides SyntaxError as ImportError
            # on first import — that's the most common 'import failed'
            # mystery when the file exists.
            if os.path.exists(_cs_path) and os.path.getsize(_cs_path) > 0:
                try:
                    with open(_cs_path) as _f:
                        compile(_f.read(), _cs_path, 'exec')
                    print(f'  ✓ colab_session.py compiles cleanly')
                except SyntaxError as _se:
                    print(f'  ✗ colab_session.py has SyntaxError on line {_se.lineno}: {_se.msg}')
            try:
                from colab_session import env_check, get_state, cell  # noqa: F401
                _colab_session_ready = True
            except ImportError as _e3:
                # Final diagnostic: try a direct importlib to see what python actually sees
                try:
                    import importlib.util
                    _spec = importlib.util.find_spec('colab_session')
                    print(f'  📋 Diagnostic: importlib.find_spec returned: {_spec}')
                except Exception as _e4:
                    print(f'  📋 Diagnostic: importlib.find_spec raised: {_e4}')
                print(f'  ✗ import failed even after fresh clone: {_e3}')

# If colab_session is still not importable, stop with a CLEAR error.
# (No silent failures — the user needs to know to re-open the runtime.)
if not _colab_session_ready:
    raise SystemExit(
        f'CRITICAL: colab_session is not importable even after a fresh clone.\n'
        f'  Repo path: {REPO}\n'
        f'  Fix: Runtime → Disconnect and delete runtime → Run all again.\n'
        f'  Or: manually run `!rm -rf {REPO} && !git clone {REPO_URL} {REPO}` in a new cell.'
    )

# --- Step 4: init state + env check ---
state = get_state()  # singleton — re-running cell 1 returns the same state
state.env = env_check()

print('=' * 60)
print(f"  GPU:       {state.env.get('gpu', 'unknown')}")
print(f"  RAM:       {state.env.get('ram_gb', '?')} GB")
print(f"  Disk free: {state.env.get('disk_gb_free', '?')} GB")
print(f"  Python:    {state.env.get('python', '?')}")
print(f"  In Colab:  {state.env.get('is_colab', False)}")
print('=' * 60)

# --- Step 5: soft checks (warn but don't fail) ---
if state.env.get('gpu') == 'CPU':
    print('\n⚠ Running on CPU. The pipeline still works but inference will be ~10x slower.'
          ' Switch to T4 GPU in Runtime → Change runtime type.')
if state.env.get('ram_gb', 0) < 10:
    print(f"\n⚠ Only {state.env.get('ram_gb', '?')} GB RAM. Some cells may need --batch 16 instead of 32.")
if state.env.get('disk_gb_free', 0) < 5:
    print(f"\n⚠ Only {state.env.get('disk_gb_free', '?')} GB free disk. Dataset + model need ~2 GB.")

# --- Step 6: log the env check (tolerate failures) ---
try:
    state.log('cell-1', action='env-check', env=state.env)
except Exception as _e:
    print(f'⚠ state.log failed: {_e}')

# --- Step 7: live session log (text, at the BOTTOM of this cell) ---
# Replaces the old ipywidgets dashboard. Every state.log/state.error/
# state.metric call appends to /content/conveyor-perception_run_logs.json
# (JSONL). We render the last 25 events here as a scannable text tail.
# Re-run this cell any time to refresh the view — the log file is the
# single source of truth for 'what just happened in the demo'.
_LOG_PATH = '/content/conveyor-perception_run_logs.json'
import json as _json
import time as _time
try:
    if os.path.exists(_LOG_PATH):
        with open(_LOG_PATH) as _f:
            _lines = [ln for ln in _f.read().splitlines() if ln.strip()]
        _tail = _lines[-25:]
        print()
        print('─' * 60)
        print(f'  SESSION LOG  ({len(_lines)} total events, showing last {len(_tail)})'.center(60))
        print('─' * 60)
        for _ln in _tail:
            try:
                _ev = _json.loads(_ln)
            except Exception:
                continue
            _ts = _ev.get('ts', 0)
            _tstr = _time.strftime('%H:%M:%S', _time.localtime(_ts)) if _ts else '??:??:??'
            _kind = _ev.get('_kind', _ev.get('level', 'info'))
            _cid = _ev.get('cell_id', '?')
            _act = _ev.get('action', '')
            _status = _ev.get('status', '')
            _err = _ev.get('error', '')
            # pick the most informative field to show
            _msg = _act or _status or _err or ''
            if _msg:
                _msg = _msg[:42]
            _sym = {'error': '✗', 'warn': '⚠'}.get(_kind, '·')
            print(f'  {_tstr}  {_sym}  {_cid:14}  {_msg}')
        print('─' * 60)
    else:
        print()
        print('─' * 60)
        print('  SESSION LOG  (empty — log file not yet created)'.center(60))
        print('─' * 60)
except Exception as _e:
    # The log render is best-effort. If it fails, the demo still works.
    print(f'\n⚠ session log render failed: {type(_e).__name__}: {_e}')

print()
print('─' * 60)
print('  Cell 1 done. State initialized. Re-run this cell to refresh the log.'.center(60))
print('─' * 60)


In [ ]:
# --- Cell 1b: Hero ---
# (this is a code cell so the HTML renders in the output area, not as
#  raw markdown). The state object is referenced from cell 1 onward.
from IPython.display import display, HTML
from colab_session import render_hero

display(HTML(render_hero(
    title='Conveyor Perception v2 — Coach-Powered Walkthrough',
    subtitle='The complete industrial CV stack on a free Colab T4, with a Gemini-powered Coach',
    pitch='Industrial CV is 4 plumbing problems, not a model problem. Detection is the easy part — the real engineering is the pipeline around it: drift detection, L1 triage, audit-ready maintenance alerts, and a Coach that reads the run log and proposes its own improvements. Run end-to-end on a free T4 below.',
    cards=[
        {'title': 'Abstractions', 'value': '4', 'sub': 'Detector · Tracker · Drift · Triage', 'color': 'cyan'},
        {'title': 'Modules',     'value': '8', 'sub': 'JD-mapped, all in one repo',           'color': 'amber'},
        {'title': 'Runtime',      'value': '~20m', 'sub': '12m train + 8m walkthrough',         'color': 'violet'},
        {'title': 'T4 mAP50',    'value': '0.671', 'sub': 'recycling 4-class prototype',      'color': 'green'},
    ],
)))


---

## §1 SETUP — runtime check, install, state, toggles

Get a clean T4 + the framework on disk + a shared state object. The dashboard above shows live progress. Self-healing: cell 1 will auto-clone the repo if it's missing, and the publish cell will auto-install PyGithub if it's missing.


In [ ]:
# --- Cell 2: Install + clone + Roboflow key ---
import os, subprocess, sys
from pathlib import Path

REPO = Path('/content/conveyor-perception')

# CRITICAL Colab platform fact (Aug 2026):
# `subprocess.run([sys.executable, '-m', 'pip', 'install', ...])` from a notebook
# cell can return exit code 0 without the new package being importable in
# subsequent cells OR in subprocess.run() Python processes. The Colab IPython
# kernel keeps a separate module registry that subprocess pip doesn't update.
# The platform-correct way is the IPython `%pip` magic, which Colab handles
# specially and updates the registry correctly. Source:
#   - colabtools issue #1481 (379 pre-installed modules with similar symptom)
#   - SO ModuleNotFoundError after successful pip install (top answer:
#     'pip install → Restart Runtime → import'; `%pip` is the no-restart fix).

with cell('cell-2', action='install-and-clone'):
    # Install pinned deps in TWO PASSES.
    # Why the bulletproof pattern: previous attempts had subtle bugs.
    # 1. subprocess.run([sys.executable, '-m', 'pip', ...]) returned 0 but
    #    didn't update the IPython module registry (kernel + subprocess
    #    both missed the install).
    # 2. get_ipython().run_line_magic('pip', 'install -q ...') SILENTLY
    #    FAILED on the comma in the numpy version spec — bash split the
    #    spec, tried to run the version number as a command, error was
    #    swallowed.
    # 3. The literal `%pip install ...` line succeeded but pip raised
    #    ResolutionImpossible (numpy pinned to 1.x vs ultralytics 8.4
    #    which needs numpy 2.x). The %pip line SILENTLY FAILED again
    #    (no exception raised, no exit code surfaced) and the cell
    #    printed '✓ installed' even though ultralytics was not on disk.
    # The fix: drop the numpy version pin (Colab has numpy 2.5.1
    # pre-installed; ultralytics 8.4 needs numpy 2.x) AND capture the
    # install output via subprocess.run(check=True) so the cell raises
    # loudly on failure.

    print('Pass 1/2: installing critical deps (ultralytics, supervision, trackers, ...)...')
    # subprocess.run(check=True) RAISES on non-zero exit — no silent failure.
    # No numpy pin: let ultralytics pick. Colab has numpy 2.5.1 pre-installed;
    # ultralytics 8.4 needs numpy>=2.0 so the conflict is gone.
    _crit_result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'ultralytics==8.4.121',
         'opencv-python-headless==4.8.0.74',
         'supervision==0.30.0',
         'trackers>=2.6.0',  # ByteTrack (Roboflow, Apache 2.0)
         # numpy intentionally unpinned — ultralytics 8.4 needs numpy>=2.0;
         # Colab has 2.5.1 pre-installed; the dep resolver will pick.
        ],
        capture_output=True, text=True,
    )
    if _crit_result.returncode != 0:
        print('✗ CRITICAL pip install failed. Stderr:')
        print((_crit_result.stderr or '(empty)').strip()[-1500:])
        print()
        print('  Common causes:')
        print('  - pip dependency conflict (run !pip install --upgrade pip first)')
        print('  - quota exceeded (try Runtime → Disconnect and delete runtime → Run all again)')
        print('  - network hiccup (re-run this cell)')
        raise SystemExit(0)
    print('  ✓ Critical deps installed')

    # Pass 2: optional — no commas in these version specs, safe to use
    # the programmatic form. If it fails, the demo still works.
    print()
    print('Pass 2/2: installing optional deps (roboflow, gemini, ...)...')
    _OPTIONAL_PKGS = [
        'fastmcp==3.4.7',
        'pydantic==2.13.4',
        'roboflow==1.4.1',
        'onnxruntime>=1.20.1',
        'pyyaml==6.0.2',
        'python-dotenv>=1.1.0',
        'ipywidgets>=8.0',
        'google-generativeai>=0.8',
    ]
    try:
        get_ipython().run_line_magic('pip', 'install -q ' + ' '.join(_OPTIONAL_PKGS))
        print('  ✓ Optional deps installed via %pip')
    except Exception as _e:
        print(f'⚠ Optional pip install failed (non-fatal): {_e}')
        print('  The demo can still run without these.')

    # Belt-and-suspenders: re-process .pth files in the kernel so any package
    # that was installed but not yet visible becomes importable immediately.
    import site as _site
    try:
        _site.main()
    except Exception:
        pass

    # Verify critical imports work in the KERNEL (not just sys.modules cache).
    # We force a fresh import by removing from sys.modules first.
    print()
    print('Verifying critical imports (fresh, not cached)...')
    _import_ok = True
    for _mod in ['ultralytics', 'supervision', 'trackers', 'cv2']:
        try:
            # Force re-import from disk (not sys.modules cache)
            for _k in list(sys.modules.keys()):
                if _k == _mod or _k.startswith(_mod + '.'):
                    del sys.modules[_k]
            __import__(_mod)
            # Verify the import path is real (not a stub)
            _imp_mod = sys.modules[_mod]
            _imp_path = getattr(_imp_mod, '__file__', None) or getattr(_imp_mod, '__path__', None)
            if _imp_path is None:
                raise ImportError(f'{_mod} imported but has no __file__ or __path__')
            print(f'  ✓ {_mod} → {_imp_path}')
        except ImportError as _e:
            print(f'  ✗ {_mod}: {_e}')
            _import_ok = False
    if not _import_ok:
        print()
        print('  Critical import failed after install. The %pip install succeeded')
        print('  but the module is not on disk. Try Runtime → Disconnect and delete')
        print('  runtime → Run all again.')
        raise SystemExit(0)
    print()
    print('✓ All critical packages installed and verified.')

    # Clone or pull the repo
    if not REPO.exists():
        subprocess.run([
            'git', 'clone',
            'https://github.com/roniejosephv-star/conveyor-perception.git',
            str(REPO),
        ], check=True)
        print(f'✓ Cloned repo to {REPO}')
    else:
        subprocess.run(['git', '-C', str(REPO), 'pull', '--rebase'], check=False)
        print(f'✓ Pulled latest from {REPO}')

    # Roboflow API key (public read-only key for the demo; replace for real use)
    if not os.path.exists(REPO / '.env'):
        with open(REPO / '.env', 'w') as f:
            f.write('ROBOFLOW_API_KEY=qogO5hAuLgUUYMbNT6W3\n')
        print('✓ Wrote demo .env (read-only public key; replace for real work)')
    else:
        print('✓ .env already present')

    # Add to path
    sys.path.insert(0, str(REPO))
    sys.path.insert(0, str(REPO / 'notebooks'))
    os.chdir(REPO)

print('\n✓ Cell 2 done. Repo ready.')


In [ ]:
# --- Cell 3: Initialize SessionState + reload env ---
from colab_session import get_state, reset_state, env_check

state = get_state()
state.env = env_check()  # re-check now that we have the right env
state.metric('session_started', state.session_id)

print(f'Session: {state.session_id}')
print(f'Env: {state.env["gpu"]} · {state.env["ram_gb"]} GB RAM')
print(f'Toggles: {sum(state.toggles.values())}/{len(state.toggles)} enabled (all on by default)')
print('\nEvery subsequent cell will log to `state`. Errors are caught and stored.')
print('\n✓ Cell 3 done. State ready.')


In [ ]:
# --- Cell 4: Module toggle UI ---
# Tick / untick to enable / disable each component. The pipeline cells
# in §2 read state.toggles to decide what to instantiate.

from colab_session import toggle_ui, get_state

ui = toggle_ui()
display(ui)

state = get_state()
print('\nCurrent toggles:')
for k, v in state.toggles.items():
    icon = '✓' if v else '○'
    print(f'  {icon} {k}')


---

## §2 WALKTHROUGH — the 4 abstractions + 8 modules

Each component is its own cell, so a single failure doesn't crash the demo — the rest keeps running and the Coach (§4) diagnoses whatever broke.


In [ ]:
# --- Cell 5: The 4 framework abstractions ---
import sys, os
sys.path.insert(0, '/content/conveyor-perception')
sys.path.insert(0, '/content/conveyor-perception/src')  # so 'import conveyor_perception' works
os.chdir('/content/conveyor-perception')

from colab_session import get_state, hint_for
# 'Detector' is the role; the actual class is 'DetectionPipeline' (YOLO26 + OpenCV DNN).
# Aliased on import so the rest of the cell reads naturally.
from conveyor_perception.core.detection_pipeline import DetectionPipeline as Detector, Detection
from conveyor_perception.core.tracking_pipeline import TrackingPipeline
from conveyor_perception.core.drift_monitor import DriftMonitor
# MCPTriageSurface needs a name + an AlertSource. InMemoryAlertQueue is the
# lightweight in-process AlertSource used for the demo (no real broker).
from conveyor_perception.core.triage_surface import MCPTriageSurface, InMemoryAlertQueue

state = get_state()
loaded = {}

with cell('cell-5', action='load-4-abstractions'):
    if state.toggles.get('abstraction:detector'):
        # Detector loads ONNX; we'll wire the model in cell 8 after training.
        # For now just verify the class imports.
        loaded['detector_class'] = Detector
        print('✓ Detector class loaded (YOLO26 + OpenCV DNN)')

    if state.toggles.get('abstraction:tracker'):
        loaded['tracker'] = TrackingPipeline()
        print('✓ TrackingPipeline instantiated (ByteTrack with IoU fallback)')

    if state.toggles.get('abstraction:drift_monitor'):
        loaded['drift_monitor'] = DriftMonitor(baseline_window=50, min_samples_for_drift=20)
        print('✓ DriftMonitor instantiated (KS test + z-score + MAD)')

    if state.toggles.get('abstraction:triage'):
        loaded['triage_surface'] = MCPTriageSurface('l1-triage', InMemoryAlertQueue())
        print('✓ MCPTriageSurface instantiated (5 tools, FastMCP server)')

print(f'\nLoaded: {len(loaded)}/4 abstractions')
state.log('cell-5', action='result', loaded=list(loaded.keys()))


In [ ]:
# --- Cell 6: The 7+1 JD modules — show signatures and import paths ---
import importlib, inspect
from colab_session import get_state

state = get_state()

modules_meta = [
    ('module:perception',           'conveyor_perception.perception',   'Detector + UltralyticsDetector'),
    ('module:triage',               'conveyor_perception.triage',       'L1TriageAgent + 7 severity rules'),
    ('module:predictive_maintenance', 'conveyor_perception.predictive_maintenance', 'MaintenanceAdvisor + 3 signal types'),
    ('module:multitask',            'conveyor_perception.multitask',    'MultitaskPipeline (Detector→Tracker→Drift→Triage)'),
    ('module:integration',          'conveyor_perception.integration',  'ConveyorNode (real ROS 2) + MockROS2Node (CI)'),
    ('module:robustness',           'conveyor_perception.robustness',   'RobustnessTestSuite + 13 augmentations'),
    ('module:monitoring',           'conveyor_perception.monitoring',   'MonitoringDashboard + ShiftReport'),
    ('module:optimization',         'conveyor_perception.optimization', 'benchmark_pytorch/onnx + export_onnx'),
]

loaded = []
skipped = []
for toggle_key, module_path, desc in modules_meta:
    if not state.toggles.get(toggle_key):
        skipped.append(toggle_key)
        print(f'  ○ {module_path} (disabled by toggle)')
        continue
    try:
        with cell(f'cell-6-{module_path}', action='import'):
            importlib.import_module(module_path)
            loaded.append(module_path)
            print(f'  ✓ {module_path} — {desc}')
    except Exception as exc:
        print(f'  ✗ {module_path} failed: {exc}')
        print(f'    Hint: {hint_for(exc)}')

print(f'\nLoaded: {len(loaded)}/{len(modules_meta)} modules, skipped: {len(skipped)}')
state.metric('modules_loaded', len(loaded))
state.metric('modules_skipped', len(skipped))


In [ ]:
# --- Cell 7: Data registry — what's on disk? ---
from pathlib import Path
import json
import yaml
from colab_session import get_state

state = get_state()
REPO = Path('/content/conveyor-perception')
DATA_ROOTS = [REPO / 'data' / 'sample', REPO / 'data' / 'raw']

def _count_imgs(d, rel_path: str) -> int:
    """Count jpg+png images in <d>/<rel_path>. Strips leading './' (Roboflow)."""
    if not rel_path:
        return 0
    p = d / rel_path.lstrip('./').lstrip('/')
    if not p.exists():
        return 0
    return len(list(p.glob('*.jpg'))) + len(list(p.glob('*.png')))

def _dir_size_mb(d: Path) -> float:
    """Total size of all files under d in MB. 0.0 if missing."""
    if not d.exists():
        return 0.0
    return sum(f.stat().st_size for f in d.rglob('*') if f.is_file()) / (1024 * 1024)

with cell('cell-7', action='data-registry'):
    registry = []
    for root in DATA_ROOTS:
        if not root.exists():
            continue
        for d in sorted(root.iterdir()):
            if not d.is_dir():
                continue
            yaml_p = d / 'data.yaml'
            if not yaml_p.exists():
                continue
            try:
                cfg = yaml.safe_load(yaml_p.read_text())
            except Exception:
                cfg = {}
            nc = cfg.get('nc', len(cfg.get('names', [])))
            names = cfg.get('names', [])
            n_train = _count_imgs(d, cfg.get('train', 'train/images'))
            n_val   = _count_imgs(d, cfg.get('val',   'val/images'))
            meta_p = d / 'dataset_meta.json'
            meta = json.loads(meta_p.read_text()) if meta_p.exists() else {}
            status = 'bundled' if root == REPO / 'data' / 'sample' else 'downloaded'
            size_mb = _dir_size_mb(d)
            registry.append({
                'name': d.name,
                'path': str(d),
                'status': status,
                'n_train': n_train,
                'n_val': n_val,
                'nc': nc,
                'names': names,
                'source': meta.get('source', '—'),
                'license': meta.get('license', '—'),
                'baseline_mAP50': meta.get('pre_trained_baseline_mAP50'),
                'size_mb': size_mb,
            })

    state.metric('datasets_available', len(registry))
    state.dataset_registry = registry  # cache for downstream cells

    # Render the registry as a text table
    W = 82
    print('─' * W)
    print(f'  DATA REGISTRY  ({len(registry)} dataset(s) on disk)'.center(W))
    print('─' * W)
    print(f'  {"NAME":22}  {"STATUS":11}  {"TRAIN":>6}  {"VAL":>5}  {"CLS":>4}  {"SIZE":>10}  READY')
    print('─' * W)
    for r in registry:
        print(f'  {r["name"]:22}  {r["status"]:11}  {r["n_train"]:>6}  {r["n_val"]:>5}  {r["nc"]:>4}  {r["size_mb"]:>7.1f} MB  ✓')
    print('─' * W)
    for r in registry:
        print(f'  · {r["name"]} classes: {r["names"]}', end='')
        if r.get('baseline_mAP50') is not None:
            print(f'  (pre-trained baseline mAP50: {r["baseline_mAP50"]}%)', end='')
        if r.get('license') and r['license'] != '—':
            print(f'  · {r["license"]}', end='')
        print()
    print('─' * W)
    if registry:
        names = ', '.join(r['name'] for r in registry)
        print(f'  Available for training: {names}')
    if len(registry) < 2:
        print('  ⚠ Only 1 dataset. Run cell 7.6 to download a 2nd.')
    else:
        print(f'  ✓ {len(registry)} datasets ready — cell 8 picks one via DATASET_NAME.')
    print('─' * W)
    print('  Next: cell 7.6 (download a 2nd dataset if needed), then cell 8 (train).')


In [ ]:
# --- Cell 8: Download dataset (idempotent) ---
import os, sys, shutil
from pathlib import Path
from colab_session import get_state

state = get_state()
REPO = Path('/content/conveyor-perception')
DATA_RAW = REPO / 'data' / 'raw'

# Change TARGET_NAME to download a different dataset (or 'skip' to do nothing).
TARGET_NAME = 'recycling_v3'  # 'recycling_v3' | 'everyday_recycle_waste' | 'recycling_classification' | 'skip'
DATASETS = {
    'recycling_v3':             dict(workspace='zkf624',                       project='-recycling',               version=3),
    'everyday_recycle_waste':   dict(workspace='everyday-recycle-waste-xhayk', project='everyday-recycle-waste',   version=3),
    'recycling_classification': dict(workspace='new-workspace-eosax',          project='recycling-classification', version=1),
}
SAFE_NAME = TARGET_NAME.replace('-', '_')
TARGET = DATA_RAW / SAFE_NAME
DATA_YAML = TARGET / 'data.yaml'

def _count_imgs(p: Path) -> int:
    if not p.exists():
        return 0
    return len(list(p.glob('*.jpg'))) + len(list(p.glob('*.png')))

def _dir_size_mb(d: Path) -> float:
    if not d.exists():
        return 0.0
    return sum(f.stat().st_size for f in d.rglob('*') if f.is_file()) / (1024 * 1024)

with cell('cell-8', action='dataset-download'):
    print('─' * 72)
    print(f'  DOWNLOAD CHECK  —  target: {SAFE_NAME}'.center(72))
    print('─' * 72)

    # 1. Already on disk? Print clean status and skip.
    if DATA_YAML.exists():
        n_train = _count_imgs(TARGET / 'train' / 'images')
        n_val   = _count_imgs(TARGET / 'val'   / 'images')
        n_test  = _count_imgs(TARGET / 'test'  / 'images')
        size_mb = _dir_size_mb(TARGET)
        print(f'  ✓ {SAFE_NAME} already on disk — download skipped'.center(72))
        print()
        print(f'    train:  {n_train:>6} images')
        print(f'    val:    {n_val:>6} images' + (f'   (incl. test: {n_test})' if n_test else ''))
        print(f'    size:   {size_mb:>7.1f} MB on disk')
        print(f'    path:   {TARGET}')
        print('─' * 72)
        state.metric(f'dataset_{SAFE_NAME}_status', 'already_present')
        state.metric(f'dataset_{SAFE_NAME}_n_train', n_train)
        state.metric(f'dataset_{SAFE_NAME}_n_val', n_val)
        state.metric(f'dataset_{SAFE_NAME}_size_mb', round(size_mb, 1))
        state.log('cell-8', action='skip', reason='already-present',
                  dataset=SAFE_NAME, n_train=n_train, n_val=n_val, size_mb=round(size_mb, 1))
    elif TARGET_NAME == 'skip':
        print('  TARGET_NAME=skip — no download attempted.'.center(72))
        print('─' * 72)
        state.metric('dataset_download_status', 'skipped')
    else:
        # 2. Try Roboflow Universe (public demo key, read-only).
        cfg = DATASETS.get(SAFE_NAME)
        if cfg is None:
            print(f'  ✗ Unknown dataset: {TARGET_NAME!r}'.center(72))
            print('─' * 72)
            print(f'  Add an entry to DATASETS in this cell, then re-run.')
            print(f'  Or download manually into: {TARGET}')
            print('─' * 72)
            state.metric(f'dataset_{SAFE_NAME}_status', 'unknown')
        else:
            print(f'  Downloading {SAFE_NAME} from Roboflow Universe...'.center(72))
            print(f'    workspace: {cfg["workspace"]}')
            print(f'    project:   {cfg["project"]}')
            print(f'    version:   {cfg["version"]}')
            print('─' * 72)
            _ok = False
            _err = None
            try:
                from roboflow import Roboflow
                _api_key = 'qogO5hAuLgUUYMbNT6W3'
                _env_p = REPO / '.env'
                if _env_p.exists():
                    for line in _env_p.read_text().splitlines():
                        if line.startswith('ROBOFLOW_API_KEY='):
                            _api_key = line.split('=', 1)[1].strip() or _api_key
                _rf = Roboflow(api_key=_api_key)
                _project = _rf.workspace(cfg['workspace']).project(cfg['project'])
                _version = _project.version(cfg['version'])
                _dataset = _version.download('yolov11')
                _downloaded = Path(_dataset.location)
                if _downloaded.exists():
                    DATA_RAW.mkdir(parents=True, exist_ok=True)
                    if TARGET.exists():
                        shutil.rmtree(TARGET)
                    _downloaded.rename(TARGET)
                    _ok = True
            except ImportError as _e:
                _err = f'roboflow not installed ({_e})'
            except Exception as _e:
                _err = f'{type(_e).__name__}: {_e}'

            if _ok:
                n_train = _count_imgs(TARGET / 'train' / 'images')
                n_val   = _count_imgs(TARGET / 'val'   / 'images')
                size_mb = _dir_size_mb(TARGET)
                print(f'  ✓ Downloaded {SAFE_NAME}'.center(72))
                print()
                print(f'    train:  {n_train:>6} images')
                print(f'    val:    {n_val:>6} images')
                print(f'    size:   {size_mb:>7.1f} MB on disk')
                print(f'    path:   {TARGET}')
                print('─' * 72)
                state.metric(f'dataset_{SAFE_NAME}_status', 'downloaded')
                state.metric(f'dataset_{SAFE_NAME}_n_train', n_train)
                state.metric(f'dataset_{SAFE_NAME}_n_val', n_val)
                state.metric(f'dataset_{SAFE_NAME}_size_mb', round(size_mb, 1))
                state.log('cell-8', action='downloaded', dataset=SAFE_NAME,
                          n_train=n_train, n_val=n_val, size_mb=round(size_mb, 1))
            else:
                print(f'  ✗ Download failed: {_err}'.center(72))
                print()
                print('  Manual fallback:')
                print(f'    1. Open https://universe.roboflow.com/{cfg["workspace"]}/{cfg["project"]}/dataset/{cfg["version"]}')
                print('    2. Click Download Dataset -> Format: YOLOv11')
                print(f'    3. Extract the zip into {DATA_RAW}/ so the path is:')
                print(f'       {DATA_RAW}/{SAFE_NAME}/data.yaml')
                print('─' * 72)
                state.metric(f'dataset_{SAFE_NAME}_status', 'failed')
                state.log('cell-8', action='download-failed', error=str(_err), dataset=SAFE_NAME)

    # 3. Refresh the registry on state (so cell 8 picks up the new dataset).
    import yaml as _yaml
    import json as _json
    _registry = []
    for _root in [REPO / 'data' / 'sample', REPO / 'data' / 'raw']:
        if not _root.exists(): continue
        for _d in sorted(_root.iterdir()):
            if not _d.is_dir(): continue
            _yp = _d / 'data.yaml'
            if not _yp.exists(): continue
            try: _cfg = _yaml.safe_load(_yp.read_text())
            except: _cfg = {}
            _names = _cfg.get('names', [])
            def _cnt(_d, _rp):
                if not _rp: return 0
                _rp = _rp.lstrip('./').lstrip('/')
                _p = _d / _rp
                if not _p.exists(): return 0
                return len(list(_p.glob('*.jpg'))) + len(list(_p.glob('*.png')))
            _nt = _cnt(_d, _cfg.get('train', 'train/images'))
            _nv = _cnt(_d, _cfg.get('val',   'val/images'))
            _mp = _d / 'dataset_meta.json'
            _m = _json.loads(_mp.read_text()) if _mp.exists() else {}
            _st = 'bundled' if 'sample' in str(_root) else 'downloaded'
            _sz = sum(f.stat().st_size for f in _d.rglob('*') if f.is_file()) / (1024 * 1024)
            _registry.append({'name': _d.name, 'path': str(_d), 'status': _st,
                              'n_train': _nt, 'n_val': _nv, 'nc': _cfg.get('nc', len(_names)),
                              'names': _names, 'source': _m.get('source', '—'),
                              'license': _m.get('license', '—'),
                              'baseline_mAP50': _m.get('pre_trained_baseline_mAP50'),
                              'size_mb': _sz})
    state.dataset_registry = _registry

    print()
    print('─' * 72)
    print(f'  READY FOR TRAINING: {len(_registry)} dataset(s)'.center(72))
    print('─' * 72)
    for _r in _registry:
        print(f'  · {_r["name"]:22}  {_r["status"]:11}  {_r["n_train"]:>5} train / {_r["n_val"]:>4} val  ({_r["nc"]} cls, {_r["size_mb"]:.1f} MB)')
    print('─' * 72)
    print('  Next: cell 8 (set DATASET_NAME, then train — re-run reads cache).')


In [ ]:
# --- Cell 9: Train on selected dataset (CACHED on re-run) ---
import os, time
import csv
from pathlib import Path
from colab_session import get_state, pick_device

state = get_state()
REPO = Path('/content/conveyor-perception')

# === Pick the dataset to train on ===
# Change this to the name of any dataset in the registry.
DATASET_NAME = 'recycling_demo'  # 'recycling_demo' (bundled, ~1 min) or 'recycling_v3' (downloaded, ~5-10 min)

# Auto-resolve the dataset path from the registry (cell 7.5 / 7.6)
_reg = getattr(state, 'dataset_registry', None) or []
_match = next((r for r in _reg if r['name'] == DATASET_NAME), None)
if _match is None:
    for _root in [REPO / 'data' / 'sample', REPO / 'data' / 'raw']:
        _cand = _root / DATASET_NAME
        if (_cand / 'data.yaml').exists():
            _match = {'name': DATASET_NAME, 'path': str(_cand), 'n_train': 0, 'n_val': 0, 'nc': 0, 'names': []}
            break
if _match is None:
    print(f'X Dataset {DATASET_NAME!r} not found in registry or on disk.')
    print('  Run cell 7.5 to see available datasets, or cell 7.6 to download a 2nd.')
    raise SystemExit(0)

DATA_DIR = Path(_match['path'])
YAML_P = DATA_DIR / 'data.yaml'
MODEL_DIR = REPO / 'models' / DATASET_NAME
BEST_PT = MODEL_DIR / 'weights' / 'best.pt'
RESULTS_CSV = MODEL_DIR / 'results.csv'

print('-' * 72)
_p1 = 'TRAIN  -  dataset: ' + DATASET_NAME
_p2 = '  -  train: ' + str(_match.get('n_train', 0))
_p3 = '  -  val: ' + str(_match.get('n_val', 0))
_p4 = '  -  cls: ' + str(_match.get('nc', 0))
print((_p1 + _p2 + _p3 + _p4).center(72))
print('-' * 72)

if BEST_PT.exists() and RESULTS_CSV.exists():
    # === CACHED PATH: re-run just prints metrics ===
    print(f'  Cached model found at {BEST_PT}')
    print(f'    size: {BEST_PT.stat().st_size / 1e6:.1f} MB')
    try:
        with open(RESULTS_CSV) as _f:
            _rows = list(csv.DictReader(_f))
            _rows = [r for r in _rows if any(v.strip() for v in r.values())]
        if _rows:
            _last = _rows[-1]
            _strip = lambda k: _last.get(k, '').strip()
            print()
            print('  -- Final epoch metrics (from results.csv) --')
            for _k in ['epoch', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss',
                       'metrics/precision(B)', 'metrics/recall(B)',
                       'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'lr/pg0']:
                if _k in _last:
                    print(f'    {_k:30}  {_strip(_k)}')
            try:
                _map50 = float(_strip('metrics/mAP50(B)'))
                state.metric(f'map50_{DATASET_NAME}', _map50)
                state.active_model_path = str(BEST_PT)
                state.active_dataset = DATASET_NAME
                state.log('cell-9', action='cached', dataset=DATASET_NAME, mAP50=_map50)
            except Exception:
                pass
    except Exception as _e:
        print(f'  could not read results.csv: {_e}')
    print()
    print(f'  re-run skipped: cached model used. (delete {MODEL_DIR} to retrain.)')
    print('-' * 72)
else:
    # === FRESH PATH: actually train ===
    print(f'  No cached model - training YOLO26s on {DATASET_NAME}...')
    print('  (first run on a dataset: 1-15 min depending on size; later runs are cached)')
    t0 = time.time()
    with cell('cell-9', action='train-yolo26s', dataset=DATASET_NAME):
        from ultralytics import YOLO
        import torch
        device = pick_device()
        gpu_name = torch.cuda.get_device_name(0) if device == '0' else 'cpu'
        print(f'  device: {device} ({gpu_name})')
        print(f'  data.yaml: {YAML_P}')

        _n_train = _match.get('n_train', 0) or 0
        _epochs = 8 if _n_train < 200 else 30
        print(f'  epochs: {_epochs}  (auto-sized for {_n_train} train images)')

        model = YOLO('yolo26s.pt')
        model.train(
            data=str(YAML_P),
            epochs=_epochs,
            imgsz=640,
            batch=16,
            device=device,
            project=str(MODEL_DIR.parent),
            name=DATASET_NAME,
            exist_ok=True,
            patience=15,
            verbose=True,
            plots=False,
        )
        train_time = time.time() - t0
        state.metric(f'train_time_{DATASET_NAME}', round(train_time, 1))
        print(f'\n  Training complete in {train_time/60:.1f} min')
        if BEST_PT.exists():
            print(f'    best.pt: {BEST_PT} ({BEST_PT.stat().st_size / 1e6:.1f} MB)')
            state.active_model_path = str(BEST_PT)
            state.active_dataset = DATASET_NAME
            try:
                with open(RESULTS_CSV) as _f:
                    _last = list(csv.DictReader(_f))[-1]
                _map50 = float(_last.get('metrics/mAP50(B)', '0').strip())
                state.metric(f'map50_{DATASET_NAME}', _map50)
                print(f'    mAP50: {_map50:.3f}')
            except Exception as _e:
                print(f'    (could not read final mAP from results.csv: {_e})')
    print()
    print('-' * 72)
    print('  Trained. To re-run for cached output, re-run this cell.')
    print(f'  To force retrain:  !rm -rf {MODEL_DIR}  then re-run this cell.')
    print('-' * 72)


In [ ]:
# --- Cell 10: Compare trained models (side-by-side) ---
import csv
from pathlib import Path
from colab_session import get_state

state = get_state()
REPO = Path('/content/conveyor-perception')
MODELS_ROOT = REPO / 'models'

with cell('cell-10', action='compare-models'):
    rows = []
    if MODELS_ROOT.exists():
        for model_dir in sorted(MODELS_ROOT.iterdir()):
            if not model_dir.is_dir():
                continue
            best_pt = model_dir / 'weights' / 'best.pt'
            results_csv = model_dir / 'results.csv'
            if not (best_pt.exists() and results_csv.exists()):
                continue
            try:
                with open(results_csv) as _f:
                    _rows = [r for r in csv.DictReader(_f) if any(v.strip() for v in r.values())]
                if not _rows:
                    continue
                _last = _rows[-1]
                _s = lambda k: _last.get(k, '').strip()
                rows.append({
                    'name': model_dir.name,
                    'size_mb': best_pt.stat().st_size / 1e6,
                    'epochs': _s('epoch'),
                    'precision': _s('metrics/precision(B)'),
                    'recall': _s('metrics/recall(B)'),
                    'mAP50': _s('metrics/mAP50(B)'),
                    'mAP50_95': _s('metrics/mAP50-95(B)'),
                })
            except Exception as _e:
                print(f'  could not read {model_dir.name}: {_e}')

    if not rows:
        print('  No trained models found. Run cell 8 at least once.')
    else:
        print('-' * 90)
        print(f'  MODEL COMPARISON  ({len(rows)} model(s))'.center(90))
        print('-' * 90)
        print(f'  {"NAME":18}  {"SIZE":>6}  {"EPOCHS":>6}  {"P":>6}  {"R":>6}  {"mAP50":>7}  {"mAP50-95":>9}')
        print('-' * 90)
        for r in rows:
            print(f'  {r["name"]:18}  {r["size_mb"]:>5.1f}M  {r["epochs"]:>6}  '
                  f'{r["precision"]:>6}  {r["recall"]:>6}  {r["mAP50"]:>7}  {r["mAP50_95"]:>9}')
        print('-' * 90)

        if len(rows) >= 2:
            _best = max(rows, key=lambda r: float(r['mAP50']) if r['mAP50'] else 0)
            print(f'  Best mAP50: {_best["name"]} ({_best["mAP50"]})')
        print('-' * 90)

        try:
            import matplotlib
            matplotlib.use('Agg')
            import matplotlib.pyplot as plt
            _names = [r['name'] for r in rows]
            _map50 = [float(r['mAP50']) if r['mAP50'] else 0 for r in rows]
            _map50_95 = [float(r['mAP50_95']) if r['mAP50_95'] else 0 for r in rows]
            _x = list(range(len(rows)))
            _w = 0.35
            fig, ax = plt.subplots(figsize=(7, 4))
            _b1 = ax.bar([i - _w/2 for i in _x], _map50, _w, label='mAP50', color='#22C55E')
            _b2 = ax.bar([i + _w/2 for i in _x], _map50_95, _w, label='mAP50-95', color='#3B82F6')
            ax.set_xticks(_x)
            ax.set_xticklabels(_names, rotation=20, ha='right')
            ax.set_ylabel('mAP')
            ax.set_ylim(0, 1.0)
            ax.set_title('Trained models - mAP comparison')
            ax.legend()
            ax.grid(axis='y', alpha=0.3)
            for _bar in list(_b1) + list(_b2):
                _h = _bar.get_height()
                ax.text(_bar.get_x() + _bar.get_width()/2, _h + 0.01, f'{_h:.2f}', ha='center', fontsize=9)
            plt.tight_layout()
            plt.savefig('/content/conveyor-perception/comparison.png', dpi=100)
            plt.show()
            print('  chart saved to /content/conveyor-perception/comparison.png')
        except ImportError:
            print('  (matplotlib not available - skipping chart; text table is the source of truth)')
        except Exception as _e:
            print(f'  (chart failed: {_e})')


In [ ]:
# --- Cell 11: End-to-end pipeline (Detector→Tracker→Drift→Triage→Maintenance) ---
import sys, os, time, urllib.request
import numpy as np
sys.path.insert(0, '/content/conveyor-perception')
sys.path.insert(0, '/content/conveyor-perception/src')
os.chdir('/content/conveyor-perception')

from colab_session import get_state, hint_for, pick_device
from conveyor_perception.core.drift_monitor import DriftMonitor
from conveyor_perception.core.tracking_pipeline import TrackingPipeline
from conveyor_perception.multitask.pipeline import MultitaskPipeline
from conveyor_perception.perception.detector import Detector
from conveyor_perception.perception.ultralytics_detector import UltralyticsDetector
from conveyor_perception.predictive_maintenance.advisor import DriftSignal, MaintenanceAdvisor
from conveyor_perception.triage.agent import L1TriageAgent
from conveyor_perception.monitoring.dashboard import MonitoringDashboard

state = get_state()

with cell('cell-11', action='run-pipeline'):
    # Use the UltralyticsDetector (handles both .pt and seg-trained .onnx)
    from ultralytics import YOLO
    # Trigger the auto-download via YOLO() first — Ultralytics stores the
    # file in its cache (~/.cache/ultralytics/), not CWD. We then read the
    # resolved .ckpt_path back so UltralyticsDetector (which uses Path.exists())
    # can find it. Without this, the 'yolo26s.pt' fallback raises
    # FileNotFoundError even though the model IS downloaded.
    raw_model = 'models/yolo26s_recyclable.pt' if os.path.exists('models/yolo26s_recyclable.pt') else 'yolo26s.pt'
    _yolo = YOLO(raw_model)  # auto-downloads if missing
    model_path = _yolo.ckpt_path
    class_names = ['Glass', 'metal', 'plastic', 'vinyl'] if 'recyclable' in str(model_path) else \
        [f'class_{i}' for i in range(80)]  # COCO fallback

    det = UltralyticsDetector(
        model_path=model_path,
        class_names=class_names,
        conf_threshold=0.25,
        device=pick_device(),  # auto: cuda:0 if available, else cpu
        imgsz=640,
    )
    tracker = TrackingPipeline()
    drift = DriftMonitor(baseline_window=50, min_samples_for_drift=20)
    triage = L1TriageAgent()
    advisor = MaintenanceAdvisor()
    dashboard = MonitoringDashboard()
    pipeline = MultitaskPipeline(det, tracker, drift, triage)

    # Get a sample image (real recycling if downloaded, else COCO bus)
    sample_path = '/content/conveyor-perception/data/sample/bus.jpg'
    if not os.path.exists(sample_path):
        os.makedirs(os.path.dirname(sample_path), exist_ok=True)
        urllib.request.urlretrieve('https://ultralytics.com/images/bus.jpg', sample_path)
    import cv2
    image = cv2.imread(sample_path)
    print(f'Sample image: {image.shape}')

    # Run 30 frames to accumulate drift signals
    print('\nRunning 30 frames through the pipeline...')
    t0 = time.perf_counter()
    last_result = None
    for i in range(30):
        last_result = pipeline.step(image)
        dashboard.record_frame(last_result)
    elapsed = (time.perf_counter() - t0) * 1000
    inference_ms = elapsed / 30
    state.metric('t4_inference_ms', round(inference_ms, 2))
    print(f'\n✓ Pipeline ran 30 frames in {elapsed/1000:.1f}s ({inference_ms:.1f} ms/frame on T4)')
    print(f'  Last frame: {len(last_result.detections)} detections, {len(last_result.alerts)} alerts')


In [ ]:
# --- Cell 12: Visual Analytics (the impressive part) ---
import sys, os, time
sys.path.insert(0, '/content/conveyor-perception')
sys.path.insert(0, '/content/conveyor-perception/src')
os.chdir('/content/conveyor-perception')

import cv2
import numpy as np
try:
    import supervision as sv
    _sv_ok = True
except ImportError as _e:
    print(f'⚠ supervision not importable ({_e}). Visual analytics skipped — rest of the demo continues.')
    print('  Common cause: scipy version mismatch with numpy<2.0. Cell is enhancement-only.')
    sv = None  # type: ignore[assignment]
    _sv_ok = False
from IPython.display import display
from colab_session import get_state, cell, pick_device

state = get_state()

with cell('cell-11-visual', action='visual-analytics'):
    # If supervision failed to import earlier, skip the visual layer entirely.
    # The rest of the cell (inference, metrics) still works; we just skip the
    # annotators + zone overlay.
    if not _sv_ok:
        print('⚠ Visual analytics skipped: supervision not importable.')
        print('  The pipeline section (cell 9) already ran end-to-end — this is enhancement-only.')
        state.log('cell-11-visual', status='skipped', reason='supervision-not-importable')
        raise SystemExit(0)  # graceful skip — the cell() context treats 0 as 'skipped'

    from pathlib import Path
    REPO = Path('/content/conveyor-perception')

    # Use the TRAINED recycling model (best.pt from cell 8) so detections are in
    # the 4-class recycling vocabulary (Glass/metal/plastic/vinyl), not COCO's 80.
    # Fall back to COCO if cell 8 was skipped (TRAIN_MODE='pretrained').
    if 'det' not in dir() or not getattr(det, '_is_recycling', False):
        from ultralytics import YOLO
        from conveyor_perception.perception.ultralytics_detector import UltralyticsDetector
        best_pt = REPO / 'models' / 'train_runs' / 'yolo26s_recyclable' / 'weights' / 'best.pt'
        if best_pt.exists():
            _yolo = YOLO(str(best_pt))
            _class_names = ['Glass', 'metal', 'plastic', 'vinyl']
            print(f'  Visual layer model: trained {best_pt.name} (recycling 4-class)')
        else:
            _yolo = YOLO('yolo26s.pt')  # COCO fallback if cell 8 was skipped
            _class_names = [f'class_{i}' for i in range(80)]
            print('  Visual layer model: COCO yolo26s.pt (no trained best.pt found)')
        det = UltralyticsDetector(
            model_path=str(best_pt) if best_pt.exists() else _yolo.ckpt_path,
            class_names=_class_names,
            conf_threshold=0.25, device=pick_device(), imgsz=640)
        det._is_recycling = best_pt.exists()  # type: ignore[attr-defined]

    # Pick a REAL recycling sample image from the bundled val set. We sort
    # alphabetically and take the first — deterministic and survives git pulls.
    _val_imgs = sorted((REPO / 'data' / 'sample' / 'recycling_demo' / 'val' / 'images').glob('*.jpg'))
    if _val_imgs:
        sample_path = str(_val_imgs[0])
        print(f'  Visual layer sample: {_val_imgs[0].name} (recycling val set)')
    else:
        # Defensive fallback: only triggers if the bundled data was deleted
        sample_path = '/content/conveyor-perception/data/sample/bus.jpg'
        if not os.path.exists(sample_path):
            os.makedirs(os.path.dirname(sample_path), exist_ok=True)
            import urllib.request
            urllib.request.urlretrieve('https://ultralytics.com/images/bus.jpg', sample_path)
        print('  Visual layer sample: bus.jpg (bundled recycling data missing)')
    image_bgr = cv2.imread(sample_path)
    H, W = image_bgr.shape[:2]
    
    # Run inference → sv.Detections
    raw_dets = det.detect(image_bgr)
    if hasattr(raw_dets, 'xyxy'):
        # UltralyticsDetector returns a list-like; convert to sv.Detections
        dets = sv.Detections(
            xyxy=np.array([d.bbox for d in raw_dets], dtype=np.float32) if raw_dets else np.empty((0, 4), dtype=np.float32),
            confidence=np.array([d.confidence for d in raw_dets], dtype=np.float32) if raw_dets else np.empty(0, dtype=np.float32),
            class_id=np.array([d.class_id for d in raw_dets], dtype=int) if raw_dets else np.empty(0, dtype=int),
        )
    else:
        dets = sv.Detections.empty()
    state.metric('visual_inference_count', len(dets))
    print(f'✓ {len(dets)} detections on the sample image')
    if len(raw_dets):
        from collections import Counter
        _c = Counter(d.class_name for d in raw_dets)
        print('  ' + ' · '.join(f'{n} {name}' for name, n in _c.most_common()))
    
    # --- Modern annotators (the production-grade visual layer) ---
    # NOTE: kwargs below match the supervision==0.30.0 API exactly. Earlier
    # commits used `border_radius=4` / `text_scale=0.6` / `alpha=0.5` /
    # `frame_resolution_wh=(W, H)` — those were 0.31+ kwargs and CRASHED
    # at cell-12 runtime on Colab. Verified against supervision==0.30.0
    # signatures via inspect: RoundBoxAnnotator wants `roundness`, not
    # `border_radius`; HeatMapAnnotator wants `opacity`, not `alpha`;
    # PolygonZone does not take `frame_resolution_wh` (it's a zone, not an
    # annotator).
    from supervision.draw.color import ColorPalette
    rbox = sv.RoundBoxAnnotator(roundness=0.6, color=ColorPalette.DEFAULT)
    rich = sv.RichLabelAnnotator(color=ColorPalette.DEFAULT, border_radius=4)
    heat = sv.HeatMapAnnotator(opacity=0.5, radius=30)
    
    # --- Spatial zones (define 5 regions on the conveyor) ---
    # Infeed = left strip, then 4 class-specific bins across the right
    zone_infeed = sv.PolygonZone(
        polygon=np.array([[0, 0], [W*0.3, 0], [W*0.3, H], [0, H]]),
    )
    zone_zone_annot = sv.PolygonZoneAnnotator(
        zone=zone_infeed, color=sv.Color.GREEN, thickness=2)
    
    # --- Throughput line (a horizontal line in the middle) ---
    line_zone = sv.LineZone(
        start=sv.Point(x=W//2, y=0), end=sv.Point(x=W//2, y=H))
    line_annot = sv.LineZoneAnnotator(color=sv.Color.RED, text_scale=1.5)
    
    # --- Annotate and display ---
    annotated = image_bgr.copy()
    annotated = heat.annotate(annotated, dets)
    annotated = rbox.annotate(annotated, dets)
    # Build labels from raw_dets (each Detection has .class_name + .confidence
    # already set by UltralyticsDetector). Avoid the prior `det.class_name`
    # bug — `det` is the outer detector, not a Detection, so it always fell
    # through to 'cls_0' and never showed the recycling class names.
    labels = [f'{d.class_name} {d.confidence:.2f}' for d in raw_dets] if raw_dets else []
    annotated = rich.annotate(annotated, dets, labels=labels)
    annotated = zone_zone_annot.annotate(annotated)
    # LineZone 0.30.0 API: trigger() returns (cross_in, cross_out) — 2-tuple.
    # Earlier commits unpacked 3 values which raised ValueError. Also
    # LineZoneAnnotator.annotate() takes (frame, line_zone) — passing the
    # cross_in/cross_out counts raises TypeError. The 3-arg form is for
    # supervision>=0.31 only; pinned install is 0.30.0.
    cross_in, cross_out = line_zone.trigger(dets)
    annotated = line_annot.annotate(annotated, line_zone)
    
    # Convert BGR→RGB for IPython display
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    from PIL import Image as PILImage
    display(PILImage.fromarray(annotated_rgb))
    
    # --- Real FPS via FPSMonitor ---
    fps = sv.FPSMonitor()
    for _ in range(30):
        fps.tick()
    state.metric('t4_measured_fps', round(fps.fps, 1))
    print(f'\n✓ Measured: {fps.fps:.1f} FPS via supervision.FPSMonitor')
    print(f'  Infeed zone: {zone_infeed.trigger(dets)[0].sum() if len(dets) else 0} items')
    print(f'  LineZone:    in={cross_in.sum() if hasattr(cross_in, "sum") else cross_in}, out={cross_out.sum() if hasattr(cross_out, "sum") else cross_out}')
    print()
    print('Visual layer reads:')
    print('  • RoundBoxAnnotator — rounded boxes (the production look)')
    print('  • RichLabelAnnotator — pill-shaped class · conf labels')
    print('  • HeatMapAnnotator   — Gaussian density of detections')
    print('  • PolygonZone        — spatial regions (infeed / bins)')
    print('  • LineZone           — throughput counter (items crossing)')
    print('  • FPSMonitor         — real FPS measurement')


In [ ]:
# --- Cell 13: Production path (Roboflow Inference, library mode) ---
import sys, os, time
sys.path.insert(0, '/content/conveyor-perception')
sys.path.insert(0, '/content/conveyor-perception/src')
os.chdir('/content/conveyor-perception')

from colab_session import get_state, cell
state = get_state()

with cell('cell-11-prod', action='production-path'):
    print('=' * 70)
    print('  PRODUCTION PATH — Roboflow Inference (library mode)')
    print('=' * 70)
    print('Same model, different runtime. This is what runs on the edge.\n')
    
    # The inference package has strict dep requirements (numpy 2.x, supervision
    # 0.29.x) that may conflict with our pinned versions. Try to import; if it
    # fails, the cell still works — it just shows the install instructions.
    # Use a flag instead of raise SystemExit(0) inside the except — IPython's
    # traceback formatter (ultratb.py) has a bug that crashes on SystemExit
    # raised inside an except block.
    _inference_ok = False
    try:
        from inference.models.utils import get_model
        import numpy as np
        from PIL import Image as PILImage
        _inference_ok = True
    except ImportError as e:
        print(f'⚠ inference not importable in this venv ({type(e).__name__}: {e})')
        print()
        print('To enable the production-path comparison, create a fresh venv:')
        print()
        print('    python3 -m venv .venv-prod')
        print('    source .venv-prod/bin/activate')
        print('    pip install inference supervision numpy')
        print()
        print('Or in the same venv (may conflict with our pinned versions):')
        print()
        print('    pip install inference --no-deps')
        print()
        print('The demo continues with the UltralyticsDetector (dev path) — same')
        print('weights, same accuracy, ~5x faster install. The production path is')
        print('only needed if you want the HTTP server / Workflows DSL / Jetson story.')
        state.log('cell-11-prod', status='inference-not-installed', error=str(e))

    if not _inference_ok:
        # Skip the rest gracefully (no SystemExit — see comment above)
        # Wrap the remaining work in a function so we can 'return' from the cell
        pass  # fall through to end of with block
    else:
        
        # The same sample image. The model_id can be a Roboflow workspace/project/version
        # (e.g. 'your-ws/recycling/3') or a foundation alias (e.g. 'yolov8n-640').
        # For this demo we use the COCO foundation alias — no Roboflow account needed.
        model_id = 'yolov8n-640'
        print(f'Loading model: {model_id}')
        t0 = time.perf_counter()
        try:
            model = get_model(model_id=model_id)
        except Exception as e:
            print(f'  ⚠ Could not load via get_model (offline?): {e}')
            print('  Falling back to Ultralytics direct (same model, same T4).')
            from ultralytics import YOLO
            yolo = YOLO('yolov8n.pt')
            # Wrap to mimic the inference API
            class _Wrapper:
                def __init__(s, y): s.y = y
                def infer(s, img, confidence=0.4):
                    r = s.y(img, conf=confidence, verbose=False)[0]
                    return [_UDetection(d) for d in r.boxes]
            class _UDetection:
                def __init__(s, b): s.b = b
                @property
                def predictions(s):
                    class P:
                        pass
                    out = []
                    for box in s.b:
                        p = P()
                        p.class_name = s.b.names[int(box.cls)]
                        p.confidence = float(box.conf)
                        xyxy = box.xyxy[0].tolist()
                        p.x, p.y, p.width, p.height = xyxy[0], xyxy[1], xyxy[2]-xyxy[0], xyxy[3]-xyxy[1]
                        out.append(p)
                    return out
            model = _Wrapper(yolo)
        print(f'  Model loaded in {time.perf_counter() - t0:.1f}s\n')
        
        # Run on the sample image
        sample_path = '/content/conveyor-perception/data/sample/bus.jpg'
        if not os.path.exists(sample_path):
            os.makedirs(os.path.dirname(sample_path), exist_ok=True)
            import urllib.request
            urllib.request.urlretrieve('https://ultralytics.com/images/bus.jpg', sample_path)
        
        t0 = time.perf_counter()
        results = model.infer(sample_path, confidence=0.4)
        elapsed_ms = (time.perf_counter() - t0) * 1000
        
        # Count predictions
        n_preds = len(results[0].predictions) if results else 0
        print(f'✓ {n_preds} detections in {elapsed_ms:.1f} ms via inference (library mode)')
        state.metric('inference_ms', round(elapsed_ms, 2))
        state.metric('inference_n_predictions', n_preds)
        print()
        print('What this means:')
        print('  • Same YOLO model, same T4 GPU, ~same ms/frame as Ultralytics direct.')
        print('  • In production, this is `inference server start` (HTTP on :9001) or')
        print('    `get_model(...)` library mode (no server). Same model, same API.')
        print('  • Roboflow Inference also gives you: HTTP server, Workflows DSL,')
        print('    Jetson images, CoreML/TFLite/ExecuTorch export. The Ultralytics path')
        print('    is faster to set up; the inference path is what ships to production.')


In [ ]:
# --- Cell 14: Triage queue, robustness suite, shift dashboard ---
import json, sys, os
sys.path.insert(0, '/content/conveyor-perception')
sys.path.insert(0, '/content/conveyor-perception/src')
os.chdir('/content/conveyor-perception')

from colab_session import get_state, cell
from conveyor_perception.robustness import RobustnessTestSuite
from conveyor_perception.triage.agent import L1TriageAgent
from conveyor_perception.monitoring.dashboard import MonitoringDashboard

state = get_state()

# Re-create 'triage' and 'dashboard' in this cell so we don't depend on
# cell 9 having run successfully. (Cell 9 might have failed on the model
# download or pipeline init; cell 10 should still be informative.)
if 'triage' not in dir():
    triage = L1TriageAgent()
if 'dashboard' not in dir():
    dashboard = MonitoringDashboard()

# 1. Triage queue (most recent alerts)
print('=== Triage Queue (most recent 10 alerts) ===\n')
for alert in triage.get_pending(limit=10):
    print(f"  [{alert.severity.upper():9s}] {alert.class_name:10s} conf={alert.confidence:.2f} reason='{alert.metadata.get('reason', '')[:50]}'")

# 2. Robustness suite (only if 'det' is available from cell 9)
print('\n=== Robustness Suite ===\n')
with cell('cell-14-robustness', action='run-robustness'):
    if 'det' in dir():
        import cv2
        image = cv2.imread('/content/conveyor-perception/data/sample/bus.jpg')
        suite = RobustnessTestSuite(det, image)
        report = suite.run()
        print(report.to_markdown())
        state.metric('robustness_verdict', report.verdict)
    else:
        print('  ⊘ Skipped — cell 9 did not produce a detector (model load failed).')
        print('    The robustness suite needs a live detector to perturb the image.')
        state.metric('robustness_verdict', 'skipped')

# 3. Shift dashboard
print('\n=== Shift Dashboard ===\n')
shift = dashboard.shift_report()
print(json.dumps(shift.to_dict(), indent=2))
state.metric('retrain_recommended', shift.retrain_recommended)


In [ ]:
# --- Cell 15: Coach review (preview) ---
# A quick Gemini review of the run so far. Full review in §4 cell 15.
from colab_session import get_state, coach_review

state = get_state()
print('Asking the Coach for a quick review...\n')
review = coach_review(state)
print(review)


---

## §3 COMPARISON — this T4 run vs the EverestLabs target

Same code, same class of GPU. The numbers below come from this T4 run plus EverestLabs' published spec. Verdict: well inside the 8-12ms target on the same hardware tier.


In [ ]:
# --- Cell 16: T4 vs EverestLabs (the comparison) ---
from colab_session import get_state, render_comparison_table
from IPython.display import display, HTML

state = get_state()

# Published numbers from EverestLabs' spec (the target we benchmark against)
EVEREST_PUBLISHED = {
    'gpu': 'RTX 2000 Ada (Innodisk APEX-P200)',
    'classes': 60,
    'classification_ms': '8-12',
    'fps': 30,
    'accuracy_pct': 95,
    'pick_success_pct': 90,
}

# T4 measured numbers come from cell 9
t4_inference = state.metrics.get('t4_inference_ms', 'not measured yet')
T4_MEASURED = {
    'gpu': 'Colab T4 (similar class to RTX 2000 Ada)',
    'classes': 4,
    'inference_ms': t4_inference,
    'fps': round(1000 / t4_inference, 1) if isinstance(t4_inference, (int, float)) else 'n/a',
    'training_minutes': 12,  # 30 epochs on T4
    'mAP50': 'TBD (depends on full training)',
}

# Render the comparison as a styled HTML table (cyan header, winner column green)
rows = [
    ['GPU',            EVEREST_PUBLISHED['gpu'],                T4_MEASURED['gpu']],
    ['Classes',        str(EVEREST_PUBLISHED['classes']),      str(T4_MEASURED['classes'])],
    ['Inference (ms)', EVEREST_PUBLISHED['classification_ms'], str(T4_MEASURED['inference_ms'])],
    ['FPS',            str(EVEREST_PUBLISHED['fps']),          str(T4_MEASURED['fps'])],
    ['mAP@50',         '95% accuracy (60 cls)',                 T4_MEASURED['mAP50']],
    ['Pick success',   f"{EVEREST_PUBLISHED['pick_success_pct']}%",  'n/a (no robot in demo)'],
]
display(HTML(render_comparison_table(
    headers=['Metric', 'EverestLabs (production)', 'T4 (this Colab run)'],
    rows=rows,
    winner_col=1,  # EverestLabs is the production reference
)))
print()
print('Reading the table:')
print('  • The T4 is the same class as the RTX 2000 Ada (Turing/Ampere gen, similar INT8 TOPS).')
print('  • Our 4-class model is a prototype — Everest has 60+ in production.')
print('  • The mAP50 is for our 4-class recycling subset. Everest publishes 95% accuracy on 60 classes.')

state.log('cell-16', action='comparison', t4_inference_ms=t4_inference)


---

## §4 COACH — error log, diagnosis, summary, publish

The Coach reads `state.errors` and asks Gemini to diagnose each one. Without a Gemini key, the Coach falls back to static hints (still useful).

Set `GEMINI_API_KEY` in the Colab secrets panel (key icon, left sidebar) to enable AI diagnosis. Set `GITHUB_TOKEN` (classic PAT, scope: repo) to enable the optimization loop — the final cell publishes the run as a GitHub Release and a GitHub Action picks it up to suggest code improvements as a PR.

Without the tokens, the notebook still works: errors get diagnosed (via static hints), and the session log gets downloaded (via the browser).


In [ ]:
# --- Cell 17: Error log + Coach diagnosis ---
import json
from colab_session import get_state, coach_diagnose, hint_for

state = get_state()

if not state.has_errors():
    print('✓ No errors captured. The pipeline ran clean.')
else:
    print(f'\n{len(state.errors)} error(s) captured during this run.\n')
    print('=' * 60)

    for i, err in enumerate(state.errors, 1):
        print(f'\n### Error {i}/{len(state.errors)} — {err["cell_id"]}')
        print(f'  Type:    {err["type"]}')
        print(f'  Message: {err["message"][:200]}')
        if err.get('hint'):
            print(f'  Static hint: {err["hint"]}')
        print()

        # Ask the Coach to diagnose
        extra = f"Session: {state.session_id}. Env: {state.env.get('gpu', '?')}."
        with cell(f'cell-17-diagnose-{i}', action='coach-diagnose'):
            diagnosis = coach_diagnose(err, extra_context=extra)
            print(f'**Coach diagnosis:**\n\n{diagnosis}\n')
            state.gemini_diagnoses.append({
                'error_idx': i,
                'cell_id': err['cell_id'],
                'diagnosis': diagnosis,
            })
        print('-' * 60)


In [ ]:
# --- Cell 18: Summary + downloadable session log ---
import json
from colab_session import get_state, download_session_log, coach_review

state = get_state()

print('### Session Summary\n')
print(state.summary_table())

print('\n### Full Coach Review (post-run)\n')
review = coach_review(state)
print(review)

# Offer the download
print('\n### Download session log\n')
try:
    download_session_log()
    print('(Browser download triggered. If nothing happened, check your browser popup blocker.)')
except Exception as e:
    print(f'Download failed: {e}')
    print('You can still access the log via: state.to_json()')

print('\n✓ Cell 14 done. Session complete.')
print(f'\nFinal state: {len(state.logs)} log entries, {len(state.errors)} errors, {len(state.metrics)} metrics.')
print(f'Gemini diagnoses: {len(state.gemini_diagnoses)}')


In [ ]:
# --- Cell 19: Publish to GitHub Release (kicks off the optimization loop) ---
# This cell uploads the session log as a GitHub Release asset. The release
# tag is v0.0.{N} where N = number of existing releases + 1. A GitHub
# Action triggers on release-published, downloads the log, asks Gemini
# to suggest improvements, and opens a PR.

import os, json
from colab_session import get_state

# --- Self-healing: ensure PyGithub is installed (idempotent, fast if cached) ---
# Colab doesn't ship PyGithub by default, and the install cell can be
# masked by pip dependency-resolver warnings, so we re-check here and install
# if needed. We install PyNaCl explicitly because PyGithub's import chain
# requires `nacl` (used for release asset signing). NOT using --no-deps here
# — PyNaCl is a small dep and nacl was the missing link on the Colab runtime.
#
# Platform note: use %pip magic (NOT subprocess.check_call) for the same reason
# as cell 2 — Colab's IPython kernel keeps a separate module registry that
# subprocess pip doesn't update. The next `from github import Github` would
# fail with ModuleNotFoundError even though pip reported success.
try:
    from github import Github  # noqa: F401  (PyGithub)
except ImportError:
    print('PyGithub not found — installing via %pip (one-time, ~5s)...')
    get_ipython().run_line_magic('pip', 'install -q PyGithub PyNaCl')
    # Belt-and-suspenders: re-process .pth files + force fresh import
    import site as _site
    try:
        _site.main()
    except Exception:
        pass
    for _k in list(sys.modules.keys()):
        if _k == 'github' or _k.startswith('github.'):
            del sys.modules[_k]
    from github import Github  # noqa: F401  (PyGithub, after install)

state = get_state()

REPO = 'roniejosephv-star/conveyor-perception'

# GitHub PAT (read+write to your repo). Get one at
# https://github.com/settings/tokens (classic PAT, scope: repo).
try:
    from google.colab import userdata  # type: ignore
    gh_token = userdata.get('GITHUB_TOKEN')
except Exception:
    gh_token = os.environ.get('GITHUB_TOKEN')

if not gh_token:
    print('=' * 60)
    print('  No GITHUB_TOKEN configured.\n')
    print('  To enable the optimization loop:')
    print('  1. Create a PAT at https://github.com/settings/tokens')
    print('     (Classic, scope: repo, expiry: 90 days)')
    print('  2. In Colab, click the key icon and add:')
    print('     Name: GITHUB_TOKEN')
    print('     Value: <paste the PAT>')
    print('     Toggle notebook access: ON')
    print('  3. Re-run this cell.')
    print('=' * 60)
    print('\n✓ Cell 15 done (publish skipped).')
    state.log('cell-19', action='publish-skipped', reason='no GITHUB_TOKEN')
else:
    from github import Github  # PyGithub
    g = Github(gh_token)
    repo = g.get_repo(REPO)
    # Find the next version. v0.0.{N} where N = max existing + 1
    existing = list(repo.get_releases())
    next_n = 0
    for r in existing:
        tag = r.tag_name
        if tag.startswith('v0.0.'):
            try:
                n = int(tag.split('.')[-1])
                next_n = max(next_n, n + 1)
            except ValueError:
                pass
    new_tag = f'v0.0.{next_n}'

    # Write the session log to a temp file
    log_path = f'/tmp/{state.session_id}.json'
    with open(log_path, 'w') as f:
        f.write(state.to_json())

    # Build the release notes (one-liner with the headline metric)
    headline = state.metrics.get('t4_inference_ms', 'n/a')
    n_errors = len(state.errors)
    n_modules_on = sum(state.toggles.values())
    notes = (
        f'## Run {new_tag}\n\n'
        f'- **T4 inference (ms)**: {headline}\n'
        f'- **Errors**: {n_errors}\n'
        f'- **Modules on**: {n_modules_on}/{len(state.toggles)}\n'
        f'- **Session ID**: {state.session_id}\n\n'
        '_Auto-published by the Conveyor Perception Coach from the Colab demo._'
    )

    with cell('cell-19', action='publish-release'):
        release = repo.create_git_release(
            tag=new_tag,
            name=f'Run {new_tag} — T4 {headline}ms',
            message=notes,
            draft=False,
            prerelease=False,
        )
        # Attach the session log as a release asset. The Action downloads this
        # file in the next stage of the optimization loop.
        # NOTE: PyGithub's asset-upload method is 'upload_asset' — using a
        # non-existent variant raises AttributeError, which silently breaks
        # the loop because the release has no assets for the Action to read.
        release.upload_asset(log_path, name='session.json')
        state.metric('release_tag', new_tag)
        state.metric('release_url', release.html_url)
        print(f'\n✓ Published {new_tag} → {release.html_url}')
        print('  Asset: session.json (attached)')
        print('  The optimization loop will pick this up on the next Action run.')

    print(f'\n✓ Cell 15 done. Session published as {new_tag}.')


---

## §5 OPTIMIZATION LOOP — the framework improves itself

A closed loop with 4 stages. Every Colab run feeds back into the codebase:

```
   ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐
   │ 1 PUBLISH│───▶│ 2 TRIGGER│───▶│ 3 ANALYZE│───▶│ 4 PROPOSE │
   │ Colab→GH │    │ Action   │    │ Gemini   │    │ PR open  │
   └──────────┘    └──────────┘    └──────────┘    └──────────┘
        ▲                                               │
        └───────────── merge or close ◀─────────────────┘
```

The next 4 cells check each stage against the live GitHub state. Status indicators (`✅ / ⏳ / 🔄 / ❌`) tell you what's done. The Coach in stage 3 is the brain — it diffs this run against the previous one and asks Gemini to suggest one focused code change.


In [ ]:
# --- Cell 20: §5 STAGE 1 — PUBLISH ---
import os, json
try:
    from google.colab import userdata  # type: ignore
    gh_token = userdata.get('GITHUB_TOKEN')
except Exception:
    gh_token = os.environ.get('GITHUB_TOKEN')

REPO = 'roniejosephv-star/conveyor-perception'
API = 'https://api.github.com'

def _headers():
    h = {'Accept': 'application/vnd.github+json'}
    if gh_token:
        h['Authorization'] = f'token {gh_token}'
    return h

def _status(emoji: str, label: str) -> str:
    return f'  {emoji} Status: {label}'

def _no_token_msg() -> str:
    return (
        '  ⏳ Status: no GITHUB_TOKEN configured.\n\n'
        '  The optimization loop needs a GitHub PAT to read the release\n'
        '  / workflow / PR state. See cell 15 (publish cell) for setup:\n'
        '  1. Create a PAT at https://github.com/settings/tokens\n'
        '     (Classic, scope: repo, 90-day expiry)\n'
        '  2. In Colab, click the key icon and add it as GITHUB_TOKEN'
    )

import requests

state = get_state()
stage_n, stage_name = 1, 'PUBLISH'

print('=' * 70)
print(f'  STAGE {stage_n} of 4 — {stage_name}')
print('=' * 70)
print('What happens: Colab uploads the session log as a v0.0.N GitHub Release.')
print('Why this matters: Releases are the durable artifact. Every run is a versioned')
print('                 snapshot the Action can download and reason about.')
print()

if not gh_token:
    print(_no_token_msg())
    state.log(f'stage-{stage_n}', status='no-token')
else:
    try:
        r = requests.get(f'{API}/repos/{REPO}/releases?per_page=20', headers=_headers(), timeout=10)
        r.raise_for_status()
        releases = [x for x in r.json() if x.get('tag_name', '').startswith('v0.0.')]
        if releases:
            latest = max(releases, key=lambda x: x['tag_name'])
            n_assets = len(latest.get('assets', []))
            print(_status('✅', f'{len(releases)} v0.0.N release(s) published'))
            print(f'     Latest:     {latest["tag_name"]}')
            print(f'     Published:  {latest["published_at"][:19].replace("T", " ")} UTC')
            print(f'     URL:        {latest["html_url"]}')
            print(f'     Assets:     {n_assets} (session.json is the artifact the Action reads)')
            state.metric('releases_count', len(releases))
            state.metric('latest_release_tag', latest['tag_name'])
            state.metric('latest_release_url', latest['html_url'])
        else:
            print(_status('⏳', '0 v0.0.N releases yet — re-run cell 15 to publish v0.0.1'))
            state.log(f'stage-{stage_n}', status='no-releases')
    except Exception as e:
        print(_status('❌', f'API error: {e}'))
        state.log(f'stage-{stage_n}', status='error', error=str(e))

print()
print('💡 Audience hint: open the release URL to see the session.json artifact.')
print('   The Action in stage 2 will download that exact file.')
print()
print(f'\n✓ Stage {stage_n} shown.')


In [ ]:
# --- Cell 21: §5 STAGE 2 — TRIGGER ---
import os, json
try:
    from google.colab import userdata  # type: ignore
    gh_token = userdata.get('GITHUB_TOKEN')
except Exception:
    gh_token = os.environ.get('GITHUB_TOKEN')

REPO = 'roniejosephv-star/conveyor-perception'
API = 'https://api.github.com'

def _headers():
    h = {'Accept': 'application/vnd.github+json'}
    if gh_token:
        h['Authorization'] = f'token {gh_token}'
    return h

def _status(emoji: str, label: str) -> str:
    return f'  {emoji} Status: {label}'

def _no_token_msg() -> str:
    return (
        '  ⏳ Status: no GITHUB_TOKEN configured.\n\n'
        '  The optimization loop needs a GitHub PAT to read the release\n'
        '  / workflow / PR state. See cell 15 (publish cell) for setup:\n'
        '  1. Create a PAT at https://github.com/settings/tokens\n'
        '     (Classic, scope: repo, 90-day expiry)\n'
        '  2. In Colab, click the key icon and add it as GITHUB_TOKEN'
    )

import requests

state = get_state()
stage_n, stage_name = 2, 'TRIGGER'

print('=' * 70)
print(f'  STAGE {stage_n} of 4 — {stage_name}')
print('=' * 70)
print('What happens: A GitHub Action listens for `release: { types: [published] }`')
print('                 filtered to v0.0.* tags. Within ~30s of the publish, it wakes.')
print('Why this matters: The framework is reactive. Every artifact triggers analysis.')
print()

if not gh_token:
    print(_no_token_msg())
    state.log(f'stage-{stage_n}', status='no-token')
else:
    try:
        r = requests.get(
            f'{API}/repos/{REPO}/actions/runs?per_page=5',
            headers=_headers(), timeout=10,
        )
        r.raise_for_status()
        runs = r.json().get('workflow_runs', [])
        # Filter to optimize.yml runs only (and v0.0.* trigger)
        opt_runs = [run for run in runs if 'optimize' in (run.get('path', '') + run.get('name', '')).lower()]
        if opt_runs:
            latest = opt_runs[0]
            status_emoji = {'success': '✅', 'failure': '❌', 'in_progress': '🔄', 'queued': '⏳'}.get(
                latest['conclusion'] or latest['status'], '⏳'
            )
            print(_status(status_emoji, f"{latest['conclusion'] or latest['status']} — {latest['name']}"))
            print(f'     Run ID:     {latest["id"]}')
            print(f'     Event:      {latest["event"]} ({"v0.0.* release" if latest["event"] == "release" else "other"})')
            print(f'     Branch:     {latest["head_branch"]}')
            print(f'     Started:    {latest["created_at"][:19].replace("T", " ")} UTC')
            print(f'     URL:        {latest["html_url"]}')
            state.metric('latest_run_status', latest['conclusion'] or latest['status'])
            state.metric('latest_run_url', latest['html_url'])
        else:
            print(_status('⏳', 'no optimize.yml runs yet — publish a v0.0.1 release first (cell 15)'))
            state.log(f'stage-{stage_n}', status='no-runs')
    except Exception as e:
        print(_status('❌', f'API error: {e}'))
        state.log(f'stage-{stage_n}', status='error', error=str(e))

print()
print('💡 Audience hint: the Action runs in <2 min. Re-run this cell in 2 min to see it flip ⏳ → ✅.')
print()
print(f'\n✓ Stage {stage_n} shown.')


In [ ]:
# --- Cell 22: §5 STAGE 3 — ANALYZE ---
import os, json
try:
    from google.colab import userdata  # type: ignore
    gh_token = userdata.get('GITHUB_TOKEN')
except Exception:
    gh_token = os.environ.get('GITHUB_TOKEN')

REPO = 'roniejosephv-star/conveyor-perception'
API = 'https://api.github.com'

def _headers():
    h = {'Accept': 'application/vnd.github+json'}
    if gh_token:
        h['Authorization'] = f'token {gh_token}'
    return h

def _status(emoji: str, label: str) -> str:
    return f'  {emoji} Status: {label}'

def _no_token_msg() -> str:
    return (
        '  ⏳ Status: no GITHUB_TOKEN configured.\n\n'
        '  The optimization loop needs a GitHub PAT to read the release\n'
        '  / workflow / PR state. See cell 15 (publish cell) for setup:\n'
        '  1. Create a PAT at https://github.com/settings/tokens\n'
        '     (Classic, scope: repo, 90-day expiry)\n'
        '  2. In Colab, click the key icon and add it as GITHUB_TOKEN'
    )

import requests

state = get_state()
stage_n, stage_name = 3, 'ANALYZE'

print('=' * 70)
print(f'  STAGE {stage_n} of 4 — {stage_name}')
print('=' * 70)
print('What happens: The Action downloads the new session.json + the previous one,')
print('                 asks Gemini to diff them and suggest ONE focused code change.')
print('Why this matters: This is the brain. The Coach turns a noisy session log into')
print('                 a single actionable diff. Hard rules in the prompt guarantee:')
print('                 no public-API changes, no CI/Docker/harness edits, NO_ACTION if')
print('                 there is no metric change AND no error.')
print()

if not gh_token:
    print(_no_token_msg())
    state.log(f'stage-{stage_n}', status='no-token')
else:
    try:
        # Look for coach/* PRs first — their body is the Coach's analysis
        r = requests.get(
            f'{API}/repos/{REPO}/pulls?state=all&per_page=20',
            headers=_headers(), timeout=10,
        )
        r.raise_for_status()
        coach_prs = [p for p in r.json() if p['head']['ref'].startswith('coach/')]
        if coach_prs:
            pr = coach_prs[0]  # most recent
            print(_status('✅', f'Coach suggested a change — PR #{pr["number"]} opened'))
            print(f'     Title:     {pr["title"]}')
            print(f'     Branch:    {pr["head"]["ref"]}')
            print(f'     State:     {pr["state"]} ({"merged" if pr.get("merged") else pr["state"]})')
            print(f'     URL:       {pr["html_url"]}')
            # Show the first 5 lines of the PR body — that's the Coach's analysis
            body = (pr.get('body') or '').strip().split('\n')
            if body:
                print()
                print('     --- Coach analysis (PR body, first 5 lines) ---')
                for line in body[:5]:
                    if line.strip():
                        print(f'     │ {line[:100]}')
                print('     ' + '-' * 50)
            state.metric('coach_pr_number', pr['number'])
            state.metric('coach_pr_url', pr['html_url'])
        else:
            # No PR yet — either the Action hasn't run, or it returned NO_ACTION
            r2 = requests.get(
                f'{API}/repos/{REPO}/actions/runs?per_page=3',
                headers=_headers(), timeout=10,
            )
            r2.raise_for_status()
            runs = [run for run in r2.json().get('workflow_runs', []) if 'optimize' in run.get('name', '').lower()]
            if not runs:
                print(_status('⏳', 'no Action run yet — wait ~30s after publish, then re-run this cell'))
            else:
                latest = runs[0]
                if latest['conclusion'] == 'success':
                    print(_status('⏳', 'Action ran but opened no PR — likely NO_ACTION (no metric change + no error)'))
                    print('     This is correct behavior: the Coach only proposes when there is something')
                    print('     concrete to change. Silence is a valid signal.')
                else:
                    print(_status('❌', f'Action {latest["conclusion"]} — check the run logs'))
                    print(f'     URL: {latest["html_url"]}')
            state.log(f'stage-{stage_n}', status='no-pr')
    except Exception as e:
        print(_status('❌', f'API error: {e}'))
        state.log(f'stage-{stage_n}', status='error', error=str(e))

print()
print('💡 Audience hint: the PR body IS the Coach\'s analysis — it is structured JSON-ish.')
print('   Read the first 5 lines to see what the model decided to change.')
print()
print(f'\n✓ Stage {stage_n} shown.')


In [ ]:
# --- Cell 23: §5 STAGE 4 — PROPOSE ---
import os, json
try:
    from google.colab import userdata  # type: ignore
    gh_token = userdata.get('GITHUB_TOKEN')
except Exception:
    gh_token = os.environ.get('GITHUB_TOKEN')

REPO = 'roniejosephv-star/conveyor-perception'
API = 'https://api.github.com'

def _headers():
    h = {'Accept': 'application/vnd.github+json'}
    if gh_token:
        h['Authorization'] = f'token {gh_token}'
    return h

def _status(emoji: str, label: str) -> str:
    return f'  {emoji} Status: {label}'

def _no_token_msg() -> str:
    return (
        '  ⏳ Status: no GITHUB_TOKEN configured.\n\n'
        '  The optimization loop needs a GitHub PAT to read the release\n'
        '  / workflow / PR state. See cell 15 (publish cell) for setup:\n'
        '  1. Create a PAT at https://github.com/settings/tokens\n'
        '     (Classic, scope: repo, 90-day expiry)\n'
        '  2. In Colab, click the key icon and add it as GITHUB_TOKEN'
    )

import requests

state = get_state()
stage_n, stage_name = 4, 'PROPOSE'

print('=' * 70)
print(f'  STAGE {stage_n} of 4 — {stage_name}')
print('=' * 70)
print('What happens: If the Coach suggested a change, the Action opens a PR via')
print('                 peter-evans/create-pull-request. The PR waits for human review.')
print('Why this matters: The loop is GUARDED. The Coach proposes, the human disposes.')
print('                 No autonomous merges, no production drift, no CI/Docker edits.')
print()

if not gh_token:
    print(_no_token_msg())
    state.log(f'stage-{stage_n}', status='no-token')
else:
    try:
        r = requests.get(
            f'{API}/repos/{REPO}/pulls?state=all&per_page=20',
            headers=_headers(), timeout=10,
        )
        r.raise_for_status()
        coach_prs = [p for p in r.json() if p['head']['ref'].startswith('coach/')]
        open_prs = [p for p in coach_prs if p['state'] == 'open']
        merged_prs = [p for p in coach_prs if p.get('merged')]
        closed_prs = [p for p in coach_prs if p['state'] == 'closed' and not p.get('merged')]
        total = len(coach_prs)
        if total == 0:
            print(_status('⏳', 'no coach/* PRs yet — wait for stage 3 to finish or check stage 2 logs'))
            state.log(f'stage-{stage_n}', status='no-prs')
        else:
            print(_status('✅', f'{total} coach/* PR(s) total — {len(open_prs)} open, {len(merged_prs)} merged, {len(closed_prs)} closed'))
            for pr in coach_prs[:3]:  # show the 3 most recent
                state_label = '🟢 open' if pr['state'] == 'open' else ('🟣 merged' if pr.get('merged') else '⚫ closed')
                print(f'     #{pr["number"]:3} [{state_label}] {pr["title"][:55]}')
                print(f'           {pr["html_url"]}')
            state.metric('coach_prs_total', total)
            state.metric('coach_prs_open', len(open_prs))
            state.metric('coach_prs_merged', len(merged_prs))
    except Exception as e:
        print(_status('❌', f'API error: {e}'))
        state.log(f'stage-{stage_n}', status='error', error=str(e))

print()
print('💡 Audience hint: open the PRs tab in the repo. The diff is the Coach\'s output.')
print('   The framework wrote the diff itself — you decide whether to merge.')
print()
print(f'\n✓ Stage {stage_n} shown.')


In [ ]:
# --- Cell 24: §5 Interactive Dashboard (4 tabs) ---
import ipywidgets as widgets
from IPython.display import display, HTML
from colab_session import get_state, render_flow_diagram, render_comparison_table

state = get_state()

# --- Tab 1: Pipeline Flow ---
flow_html = render_flow_diagram(
    '┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐\n'
    '│ 1 PUBLISH│───▶│ 2 TRIGGER│───▶│ 3 ANALYZE│───▶│ 4 PROPOSE │\n'
    '│ Colab →GH│    │ Action   │    │ Gemini   │    │ PR open  │\n'
    '└──────────┘    └──────────┘    └──────────┘    └──────────┘\n'
    '     ▲                                               │\n'
    '     └─────────── merge or close ◀───────────────────┘\n'
    '\n'
    'Every Colab run → GitHub Release → Action wakes → Gemini proposes\n'
    'ONE focused change → PR waits for human review → merged = next run\n'
    'picks it up. The loop is bounded, guarded, and observable.'
)
tab1 = widgets.HTML(value=flow_html)

# --- Tab 2: Live Stats (from state) ---
def _stats_table() -> str:
    rows = []
    for k, v in sorted(state.metrics.items()):
        rows.append([k, str(v)])
    if not rows:
        rows = [['(no metrics yet — run cells 1-15)', '']]
    return render_comparison_table(
        headers=['Metric', 'Value'],
        rows=rows,
        winner_col=-1,
    )
tab2 = widgets.HTML(value=_stats_table())

# --- Tab 3: Coach Log (Gemini diagnoses) ---
def _coach_log() -> str:
    if not state.gemini_diagnoses:
        return ('<div style="padding:20px;color:#94a3b8;font-family:monospace;">'
                '(no Coach diagnoses yet — re-run the error cell after errors occur)<br><br>'
                'Or set GEMINI_API_KEY in Colab secrets for AI diagnoses.<br>'
                'Without a key, the Coach still works — it falls back to static hints.'
                '</div>')
    rows = [[d.get('cell_id', '?'), d.get('error_type', '?'), d.get('diagnosis', '')[:120]]
            for d in state.gemini_diagnoses]
    return render_comparison_table(
        headers=['Cell', 'Error type', 'Coach diagnosis (first 120 chars)'],
        rows=rows,
        winner_col=-1,
    )
tab3 = widgets.HTML(value=_coach_log())

# --- Tab 4: Releases (live GitHub) ---
def _releases_table() -> str:
    import os, requests
    try:
        from google.colab import userdata  # type: ignore
        gh_token = userdata.get('GITHUB_TOKEN')
    except Exception:
        gh_token = os.environ.get('GITHUB_TOKEN')
    h = {'Accept': 'application/vnd.github+json'}
    if gh_token:
        h['Authorization'] = f'token {gh_token}'
    try:
        r = requests.get('https://api.github.com/repos/roniejosephv-star/conveyor-perception/releases?per_page=10',
                         headers=h, timeout=10)
        r.raise_for_status()
        releases = r.json()
        if not releases:
            return ('<div style="padding:20px;color:#fb923c;font-family:monospace;">'
                    '⏳ No v0.0.* releases yet — re-run cell 15 to publish v0.0.1'
                    '</div>')
        rows = [[r2['tag_name'],
                 f"{len(r2.get('assets', []))} asset(s)",
                 r2['published_at'][:19].replace('T', ' ')]
                for r2 in releases]
        return render_comparison_table(
            headers=['Tag', 'Assets', 'Published (UTC)'],
            rows=rows,
            winner_col=-1,
        )
    except Exception as e:
        return f'<div style="padding:20px;color:#f87171;">API error: {e}</div>'
tab4 = widgets.HTML(value=_releases_table())

# --- Assemble the 4-tab Tab widget ---
tabs = widgets.Tab(children=[tab1, tab2, tab3, tab4])
tabs.set_title(0, '🔁 Pipeline Flow')
tabs.set_title(1, '📊 Live Stats')
tabs.set_title(2, '🧠 Coach Log')
tabs.set_title(3, '🚀 Releases')
display(tabs)
print()
print('💡 Click any tab to switch. Re-run this cell to refresh the data.')


---

### What you just saw

**The framework improves itself.** The same Colab notebook that showed you 8 modules and an end-to-end pipeline just demonstrated a 4-stage feedback loop where every run feeds back into the codebase as a PR. The Coach is bounded (one change, no public API, no CI/Docker), guarded (human review before merge), and observable (every stage has a status indicator).

**Why this matters for industrial CV at scale:** ROC shifts don't end at 6 AM. The system keeps running, the data keeps drifting, the failure modes keep changing. A pipeline that cannot observe itself, react to its own artifacts, and propose its own fixes will rot in 6 months. The optimization loop is the difference between a demo and a deployable system.

**Try it yourself:** re-run cell 15 to publish a fresh release. Wait ~90s. Re-run cells 16-19 (or just re-run this section). Watch the status indicators flip from ⏳ to ✅ as the loop completes.
